In [1]:
ALL_METHODS = [
    "results___NPBIR___EST_g__EST_l",  # OG
    "results___NPBIR____GT_g__EST_l",
    "results___NPBIR___EST_g___GT_l",
    "results___NPBIR__VGGT_g__EST_l",
    "results___NPBIR__VGGT_g___GT_l",
]


METHOD=ALL_METHODS[3]
NPBIR="./DigitalTwinCatalog/neural_pbir"
CHKPT=f"./{METHOD}/stanford_orb"
SORB = "/home/ahc/Datasets/Stanford-ORB"
SORB_HDR = f"{SORB}/blender_HDR"
SORB_GT = f"{SORB}/ground_truth"
SCENE = "teapot_scene001" # "baking_scene001"

In [11]:
# analyze mesh alignment first
M1="/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj"
M2="/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj"
M3="/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_blender/mesh.obj" 
!python compare_mesh_alignment.py {M1} {M2} {M3} --output aligned_comparison_vggt.ply

Loading Red mesh from: /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj

Per mesh stats [/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj]:
  Total vertices: 32592
  Total faces: 64592
  Bounds: [[-0.306375 -0.236633 -0.182005]
 [ 0.242847  0.277442  0.320402]]
Loading Green mesh from: /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj

Per mesh stats [/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj]:
  Total vertices: 32592
  Total faces: 64592
  Bounds: [[-0.306375 -0.236633 -0.182005]
 [ 0.242847  0.277442  0.320402]]
Loading Blue mesh from: /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_blender/mesh.obj

Per mesh stats [/home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_blender/mesh.obj]:
  Total vertices: 118976
  Total faces: 200000
  Bounds: [[-0.27192304 -0.21533936 -0.18616989]
 [ 0.37355959  0.23565942  0.31300035]]

Combined mesh stats:
  T

In [10]:
!python ~/Documents/metrology_ir/vggt_geometry.py {SORB_HDR}/{SCENE} --max_images 18 --wnnc_width_config l5 --hdr --output {SORB_GT}/{SCENE}/mesh_vggt.obj --align_voxel_fraction 0.01 --align_to {SORB_GT}/{SCENE}/mesh_blender/mesh.obj

Using 18 image/mask pairs from 'train' split
Loading VGGT-1B...
Preprocessing images...
Running VGGT forward pass...
/home/ahc/Documents/metrology_ir/vggt_geometry.py:312: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):
/home/ahc/Documents/metrology_ir/vggt/vggt/models/vggt.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Filtering points by mask and confidence...
  325,258 points before downsampling
  36,792 points after downsampling/outlier removal
  Saved -> /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/pointcloud_vggt.ply
Running WNNC (width_config=l5)...
num_nodes: 51433, num_leaves: 36792, tree depth: 10
tree_depth*num_points: 367920, stdvec_node2point_index.size(): 299557
  Normals -> /home/ahc/Datasets/Stanford-ORB/grou

In [ ]:
# run once per new scene within the dataset itself (regardless of method, etc.)
# !python DigitalTwinCatalog/neural_pbir/scripts/preprocess/stanford_orb.py /home/ahc/Datasets/Stanford-ORB/blender_HDR/{SCENE}

!python {NPBIR}/neural_surface_recon/run_template.py --template {NPBIR}/neural_surface_recon/configs/template_stanford_orb.py --savemem {SORB_HDR}/{SCENE}/ --mesh_init_path={SORB_GT}/{SCENE}/mesh_vggt.obj
!mv results {METHOD}

!python {NPBIR}/neural_distillation/run.py {CHKPT}/{SCENE}/
!python {NPBIR}/pbir/run.py {NPBIR}/pbir/configs/template {CHKPT}/{SCENE}/
!python {NPBIR}/scripts/stanford_orb/render_geo.py {SORB_HDR}/{SCENE}/cameras.json {CHKPT}/{SCENE}/pbir/mesh.obj

!python {NPBIR}/scripts/stanford_orb/postproc_envmap_our.py "{CHKPT}/{SCENE}/pbir/envmap.exr"

# NOVEL VIEW SYNTHESIS
CAM=f"{SORB_HDR}/{SCENE}/cameras.json"
CKPT_GEO=f"{CHKPT}/{SCENE}/pbir/mesh.obj"
CKPT_ALBEDO=f"{CHKPT}/{SCENE}/pbir/diffuse.exr"
CKPT_ROUGH=f"{CHKPT}/{SCENE}/pbir/roughness.exr"
CKPT_ENV=f"{CHKPT}/{SCENE}/pbir/envmap_for_blender.exr"
!python {NPBIR}/scripts/relit/relit.py {CKPT_GEO} {CKPT_ALBEDO} {CKPT_ROUGH} {CAM} --lgt_paths {CKPT_ENV} --render_exr --with_bg

# RELIGHTING
!python {NPBIR}/scripts/stanford_orb/relit_nl.py --ckptroot {CHKPT} --dataroot {SORB}


!python {NPBIR}/scripts/stanford_orb/eval_preprocess.py --data_dir {SORB}/  --ckpt_dir {CHKPT}/
!cd Stanford-ORB && PYTHONPATH=. python scripts/test.py --input-path ../{CHKPT}/eval_inputs_pbir.json --output-path ../{CHKPT}/eval_outputs_pbir.json --scenes example


/home/ahc/miniconda3/envs/metrology_ir/lib/python3.11/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
==> disable pre-load all training data to save GPU memory
==> using mesh prior for SDF init: /home/ahc/Datasets/Stanford-ORB/ground_truth/teapot_scene001/mesh_vggt.obj
/home/ahc/miniconda3/envs/metrology_ir/lib/python3.11/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
Start loading dataset...